<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E5_Caso_End_to_End_Wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E5 · Caso end-to-end de clustering - Análisis cluster (bonus)

## Introducción

Un caso completo, de principio a fin: **preparar** los datos, **comparar** los tres métodos que
hemos visto (K-Means, jerárquico y DBSCAN), **elegir** una solución, **perfilar** los grupos y
**traducir** cada uno a una acción.

> El entregable final no es la tabla: es el **método elegido + métricas + perfiles +
> recomendación**.

## Objetivos del ejercicio

- Preparar y escalar un dataset real.
- Comparar **K-Means, jerárquico y DBSCAN** con el Silhouette Score.
- Elegir solución, **perfilar** los grupos y dar una **recomendación accionable**.

## Descripción del dataset (Wine)

**Wine** (sklearn) es real: 178 vinos descritos por 13 propiedades químicas (alcohol, fenoles,
color...), procedentes de 3 cultivos distintos. Lo trataremos como no supervisado (sin mirar el
cultivo) y al final comprobaremos si los grupos coinciden con los 3 cultivos reales.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

### 2. Preparar y escalar

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target              # cultivo real (solo para validar al final)
X_esc = StandardScaler().fit_transform(X)
print("Forma:", X.shape, "| cultivos reales:", len(np.unique(y)))

### 3. Comparar métodos (Silhouette)

In [ ]:
resultados = {
    "K-Means (k=3)": KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X_esc),
    "Jerárquico Ward (k=3)": AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X_esc),
    "DBSCAN (eps=2.5)": DBSCAN(eps=2.5, min_samples=5).fit_predict(X_esc),
}

for nombre, lab in resultados.items():
    n_cl = len(set(lab) - {-1})
    ruido = int((lab == -1).sum())
    if n_cl >= 2:
        mask = lab != -1
        sil = silhouette_score(X_esc[mask], lab[mask])
        print(f"{nombre:<24} -> {n_cl} clusters | ruido {ruido:>3} | silhouette {sil:.3f}")
    else:
        print(f"{nombre:<24} -> {n_cl} cluster(s) | ruido {ruido:>3} | silhouette N/A")

DBSCAN suele quedarse corto aquí: los grupos del vino son compactos y de densidad parecida, no
hay "zonas densas separadas por vacíos". Es un buen recordatorio: **no hay un único clustering**,
cada método encaja con un tipo de problema. Nos quedamos con la solución de mejor silhouette
(K-Means o jerárquico).

### 4. Elegir solución y validar con el cultivo real

In [ ]:
labels = resultados["K-Means (k=3)"]
print(f"Silhouette: {silhouette_score(X_esc, labels):.3f}")
print(f"Adjusted Rand Index vs cultivo real: {adjusted_rand_score(y, labels):.3f}  (1.0 = idéntico)")

X_2d = PCA(n_components=2, random_state=0).fit_transform(X_esc)
plt.figure(figsize=(7, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="tab10", s=25)
plt.title("Solución elegida (K-Means, k=3) vista en 2D con PCA")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.show()

### 5. Perfilar los grupos (media vs media global)

In [ ]:
dfw = X.copy()
dfw["cluster"] = labels
indice = (dfw.groupby("cluster").mean() / X.mean()).round(2)
# Las variables que mas separan se miden en escala ESTANDARIZADA (si no, mandan las de
# mayor magnitud como 'proline' solo por su escala, no por separar de verdad).
dfe = pd.DataFrame(X_esc, columns=X.columns)
dfe["cluster"] = labels
discrimina = dfe.groupby("cluster").mean().std().sort_values(ascending=False)
top = discrimina.head(4).index.tolist()
print("Variables que más separan los grupos (escala estandarizada):", top)
print("\nÍndice vs media global (1.00 = como el vino medio):")
indice[top]

### 6. Recomendación (de números a decisión)

Traducimos cada grupo a una etiqueta y una acción. Imagina una **bodega/distribuidor** que quiere
organizar su catálogo por estilos de vino.

In [ ]:
resumen = dfw.groupby("cluster")[top].mean().round(2)
resumen["n_vinos"] = dfw["cluster"].value_counts().sort_index()
print("Perfil de cada grupo (variables más discriminantes):")
print(resumen)
print("\nEjemplo de lectura accionable (ajústala a tus números):")
print("  - Grupo con más alcohol/fenoles -> 'vinos intensos' -> gama premium / guarda.")
print("  - Grupo más suave -> 'vinos ligeros' -> consumo diario / volumen.")
print("\nEntregable: método (K-Means k=3) + silhouette + perfiles + acción por grupo.")

### Reflexión

1. ¿Por qué DBSCAN no brilla en este dataset?
2. ¿Coinciden los grupos hallados con los 3 cultivos reales (mira el ARI)?
3. ¿Qué 3-4 variables bastan para describir los grupos?
4. Si tuvieras que presentar esto a negocio, ¿cuál sería tu recomendación en una frase?